# CNN-EXP-006 — Local + Wide Global BPE

Экспериментальная двухветочная модель:

- **LOCAL:** 1284 bp → base-resolution CNN.
- **GLOBAL:** 9216 bp → BPE-50 → dilated CNN.
- Global-признаки выравниваются обратно к центральным 1284 bp и объединяются с local-признаками.
- Выход: две головы — `presence` (`count > 0`) и `intensity` (`log1p(count)`).

До раздела обучения оставлены только необходимые определения и подготовка данных.


In [1]:
from pathlib import Path
import sys
import gc

from IPython.display import display
import torch
import torch.nn as nn
import torch_directml

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = next(
    candidate for candidate in (CURRENT_DIR, *CURRENT_DIR.parents)
    if (candidate / "README.md").is_file()
    and (candidate / "src/ml_project").is_dir()
)

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from ml_project.epic_baseline import load_baseline_sequences
from ml_project.epic_cnn import (
    DirectMLAdam,
    ProfileWindowConfig,
    ProfileWindowGenerator,
    create_cnn_presence_intensity,
    dml_weighted_huber,
    dml_weighted_profile_bce_with_logits,
    forward_both_strands_multitask,
)
from ml_project.epic_data import load_split

print("Project:", PROJECT_ROOT)


Project: D:\Projects\EPIC\EPIC


In [2]:
device = torch_directml.device()

print("DirectML device:", device)
print("GPU:", torch_directml.device_name(0))


DirectML device: privateuseone:0
GPU: AMD Radeon RX 6750 GRE 12GB 


In [3]:
split = load_split(PROJECT_ROOT)
sequences, fasta_alphabet = load_baseline_sequences(split)

train_contigs = set(split.intervals("train")["contig"])
validation_contigs = set(split.intervals("validation")["contig"])

assert train_contigs.isdisjoint(validation_contigs)

display(split.summary)
print("FASTA contigs:", len(sequences))
print("Train contigs:", len(train_contigs))
print("Validation contigs:", len(validation_contigs))


,split,contigs,intervals_both_strands,whitelist_bp,position_strand_count,percent_of_all_positions,positive_positions,negative_positions,positive_percent
0,train,9,168998,114032080,228064160,59.316784,170443.0,227893717.0,0.074735
1,validation,3,60386,41903743,83807486,21.797334,56205.0,83751281.0,0.067064
2,test,3,54846,36306695,72613390,18.885882,NaN,NaN,NaN


FASTA contigs: 47
Train contigs: 9
Validation contigs: 3


In [4]:
profile_config = ProfileWindowConfig(
    target_length=1024,
    context_flank=130,
    positive_branch_fraction=0.5,
)

train_generator = ProfileWindowGenerator(
    prepared_split=split,
    sequences=sequences,
    part="train",
    config=profile_config,
    seed=42,
)

train_batch = train_generator.sample_batch(batch_size=4)

print("Local input:", train_batch.x.shape)
print("Both orientations:", train_batch.x_both_strands.shape)
print("Target:", train_batch.target.shape)
print("Count:", train_batch.count.shape)


Local input: torch.Size([4, 5, 1284])
Both orientations: torch.Size([8, 5, 1284])
Target: torch.Size([4, 2, 1024])
Count: torch.Size([4, 2, 1024])


In [5]:
# BPE-50 artifact + torch-only tokenizer

BPE_PATH = PROJECT_ROOT / "artifacts" / "bpe_softmask_50.pt"

bpe_artifact = torch.load(
    BPE_PATH,
    map_location="cpu",
    weights_only=False,
)

merge_history = bpe_artifact["merge_history"]
token_text = bpe_artifact["token_text"]
token_length = bpe_artifact["token_length"]
vocab_size = bpe_artifact["vocab_size"]
PAD_TOKEN_ID = vocab_size

# Genome storage codes:
# 0..3 = A,C,G,T
# 4..7 = a,c,g,t
# 8    = N/other
BASE_LOOKUP = torch.tensor(
    [0, 1, 2, 3, 0, 1, 2, 3, 4],
    dtype=torch.long,
)

SOFTMASK_LOOKUP = torch.tensor(
    [0, 0, 0, 0, 1, 1, 1, 1, 0],
    dtype=torch.long,
)

RC_LOOKUP = torch.tensor(
    [3, 2, 1, 0, 7, 6, 5, 4, 8],
    dtype=torch.long,
)

token_length_lookup = torch.zeros(
    vocab_size,
    dtype=torch.long,
)

for token_id, length in token_length.items():
    token_length_lookup[int(token_id)] = int(length)


def apply_bpe_torch(
    sequence: torch.Tensor,
    lowercase_count: torch.Tensor,
    merge_rules,
):
    """Apply learned BPE merges using torch tensors only."""
    tokens = sequence.clone()
    lower = lowercase_count.clone()

    for left, right, new_token in merge_rules:
        left = int(left)
        right = int(right)
        new_token = int(new_token)

        matches = (
            (tokens[:-1] == left)
            & (tokens[1:] == right)
        )

        starts = torch.nonzero(
            matches,
            as_tuple=False,
        ).flatten()

        if starts.numel() == 0:
            continue

        # Only X+X can produce overlapping adjacent pair starts.
        if left == right and starts.numel() > 1:
            new_run = torch.cat(
                [
                    torch.tensor(
                        [True],
                        dtype=torch.bool,
                        device=starts.device,
                    ),
                    starts[1:] != starts[:-1] + 1,
                ]
            )

            run_ids = torch.cumsum(
                new_run.to(torch.long),
                dim=0,
            ) - 1

            run_start_indices = torch.nonzero(
                new_run,
                as_tuple=False,
            ).flatten()

            start_index_for_each_run = (
                run_start_indices[run_ids]
            )

            offsets = (
                torch.arange(
                    starts.numel(),
                    device=starts.device,
                )
                - start_index_for_each_run
            )

            starts = starts[offsets % 2 == 0]

        new_tokens = tokens.clone()
        new_tokens[starts] = new_token

        new_lower = lower.clone()
        new_lower[starts] = (
            lower[starts]
            + lower[starts + 1]
        )

        remove = torch.zeros(
            tokens.shape[0],
            dtype=torch.bool,
            device=tokens.device,
        )
        remove[starts + 1] = True

        tokens = new_tokens[~remove]
        lower = new_lower[~remove]

    return tokens, lower


def encode_raw_bpe(raw: torch.Tensor):
    collapsed = BASE_LOOKUP[raw]
    softmask = SOFTMASK_LOOKUP[raw]

    tokens, lower_count = apply_bpe_torch(
        collapsed,
        softmask,
        merge_history,
    )

    lengths = token_length_lookup[tokens]

    softmask_fraction = (
        lower_count.float()
        / lengths.float()
    )

    bp_to_token = torch.repeat_interleave(
        torch.arange(
            tokens.numel(),
            dtype=torch.long,
        ),
        lengths,
    )

    assert bp_to_token.numel() == raw.numel()

    return {
        "tokens": tokens,
        "lengths": lengths,
        "softmask_fraction": softmask_fraction,
        "bp_to_token": bp_to_token,
    }


print(
    "BPE:",
    f"{len(merge_history)} merges,",
    f"vocab={vocab_size},",
    f"pad={PAD_TOKEN_ID}",
)


BPE: 50 merges, vocab=55, pad=55


In [6]:
# Wide global window + padded BPE batches

GLOBAL_FLANK = 4096
LOCAL_FLANK = profile_config.context_flank
TARGET_LENGTH = profile_config.target_length

GLOBAL_LENGTH = (
    GLOBAL_FLANK
    + TARGET_LENGTH
    + GLOBAL_FLANK
)

LOCAL_LENGTH = (
    LOCAL_FLANK
    + TARGET_LENGTH
    + LOCAL_FLANK
)

LOCAL_START_IN_GLOBAL = (
    GLOBAL_FLANK
    - LOCAL_FLANK
)

LOCAL_END_IN_GLOBAL = (
    LOCAL_START_IN_GLOBAL
    + LOCAL_LENGTH
)


def extract_raw_window_torch(
    contig,
    start,
    end,
):
    """Extract exact genomic window and pad outside contig with N code 8."""
    contig_seq = sequences[contig]
    contig_length = len(contig_seq)
    wanted_length = end - start

    clipped_start = max(start, 0)
    clipped_end = min(end, contig_length)

    core = torch.as_tensor(
        contig_seq[clipped_start:clipped_end],
        dtype=torch.long,
    ).clone()

    left_pad = max(0, -start)
    right_pad = max(0, end - contig_length)

    if left_pad:
        core = torch.cat(
            [
                torch.full(
                    (left_pad,),
                    8,
                    dtype=torch.long,
                ),
                core,
            ]
        )

    if right_pad:
        core = torch.cat(
            [
                core,
                torch.full(
                    (right_pad,),
                    8,
                    dtype=torch.long,
                ),
            ]
        )

    assert core.numel() == wanted_length
    return core


def build_wide_global_bpe_batch(
    tiles,
    reverse_complement=False,
):
    """Build CPU BPE tensors for one orientation of a tile batch."""
    encoded = []

    for i in range(len(tiles)):
        tile = tiles.iloc[i]

        contig = str(tile["contig"])
        target_start = int(tile["target_start"])
        target_end = int(tile["target_end"])

        raw = extract_raw_window_torch(
            contig,
            target_start - GLOBAL_FLANK,
            target_end + GLOBAL_FLANK,
        )

        if reverse_complement:
            raw = RC_LOOKUP[raw].flip(0)

        item = encode_raw_bpe(raw)

        item["local_bp_to_token"] = item["bp_to_token"][
            LOCAL_START_IN_GLOBAL:
            LOCAL_END_IN_GLOBAL
        ]

        encoded.append(item)

    batch_size = len(encoded)
    max_tokens = max(
        item["tokens"].numel()
        for item in encoded
    )

    token_ids = torch.full(
        (batch_size, max_tokens),
        PAD_TOKEN_ID,
        dtype=torch.long,
    )

    # These three are float32 already on CPU.
    # That avoids bool/int -> float arithmetic inside DirectML.
    token_lengths = torch.zeros(
        (batch_size, max_tokens),
        dtype=torch.float32,
    )

    softmask = torch.zeros(
        (batch_size, max_tokens),
        dtype=torch.float32,
    )

    token_mask = torch.zeros(
        (batch_size, max_tokens),
        dtype=torch.float32,
    )

    local_bp_to_token = torch.zeros(
        (batch_size, LOCAL_LENGTH),
        dtype=torch.long,
    )

    for i, item in enumerate(encoded):
        n = item["tokens"].numel()

        token_ids[i, :n] = item["tokens"]
        token_lengths[i, :n] = item["lengths"].float()
        softmask[i, :n] = item["softmask_fraction"]
        token_mask[i, :n] = 1.0
        local_bp_to_token[i] = item["local_bp_to_token"]

    return {
        "token_ids": token_ids,
        "token_lengths": token_lengths,
        "softmask": softmask,
        "token_mask": token_mask,
        "local_bp_to_token": local_bp_to_token,
    }


# Current batch stays on CPU until a forward pass needs it.
global_plus_batch = build_wide_global_bpe_batch(
    train_batch.tiles,
    reverse_complement=False,
)

global_rc_batch = build_wide_global_bpe_batch(
    train_batch.tiles,
    reverse_complement=True,
)

print(
    "Wide BPE PLUS / RC:",
    global_plus_batch["token_ids"].shape,
    global_rc_batch["token_ids"].shape,
)


Wide BPE PLUS / RC: torch.Size([4, 3708]) torch.Size([4, 3709])


In [7]:
# Local + Wide Global BPE model
# DirectML-safe masking: masks are always cast to activation dtype.
# Gather indices are made contiguous before torch.gather.

GLOBAL_DILATIONS = (
    1, 2, 4, 8, 16,
    32, 64, 128, 256, 512,
)


class GlobalResidualBlock(nn.Module):

    def __init__(
        self,
        channels: int,
        dilation: int,
    ):
        super().__init__()

        self.conv1 = nn.Conv1d(
            channels,
            channels,
            kernel_size=3,
            dilation=dilation,
            padding=dilation,
        )

        self.conv2 = nn.Conv1d(
            channels,
            channels,
            kernel_size=3,
            dilation=dilation,
            padding=dilation,
        )

        self.activation = nn.GELU()

    def forward(
        self,
        x,
        token_mask,
    ):
        mask = (
            token_mask
            .unsqueeze(1)
            .to(dtype=x.dtype)
        )

        residual = x

        x = self.conv1(x)
        x = self.activation(x)
        x = x * mask

        x = self.conv2(x)
        x = x + residual
        x = self.activation(x)
        x = x * mask

        return x


class GlobalBPEEncoder(nn.Module):

    def __init__(
        self,
        vocab_size: int,
        pad_token_id: int,
        embedding_dim: int = 16,
        channels: int = 32,
        max_token_length: int = 4,
        dilations=GLOBAL_DILATIONS,
    ):
        super().__init__()

        self.max_token_length = max_token_length
        self.dilations = tuple(dilations)

        self.embedding = nn.Embedding(
            vocab_size + 1,
            embedding_dim,
            padding_idx=pad_token_id,
        )

        self.input_projection = nn.Sequential(
            nn.Conv1d(
                embedding_dim + 2,
                channels,
                kernel_size=1,
            ),
            nn.GELU(),
        )

        self.blocks = nn.ModuleList(
            [
                GlobalResidualBlock(
                    channels=channels,
                    dilation=d,
                )
                for d in self.dilations
            ]
        )

    def forward(
        self,
        token_ids,
        token_lengths,
        softmask_fraction,
        token_mask,
    ):
        embedding = self.embedding(token_ids)

        length_feature = (
            token_lengths
            .to(dtype=embedding.dtype)
            .div(self.max_token_length)
            .unsqueeze(-1)
        )

        softmask_feature = (
            softmask_fraction
            .to(dtype=embedding.dtype)
            .unsqueeze(-1)
        )

        x = torch.cat(
            [
                embedding,
                softmask_feature,
                length_feature,
            ],
            dim=-1,
        )

        mask_bt = (
            token_mask
            .unsqueeze(-1)
            .to(dtype=x.dtype)
        )

        x = x * mask_bt
        x = x.transpose(1, 2).contiguous()

        x = self.input_projection(x)

        mask_bct = (
            token_mask
            .unsqueeze(1)
            .to(dtype=x.dtype)
        )
        x = x * mask_bct

        for block in self.blocks:
            x = block(x, token_mask)

        return x


class CNNPresenceIntensityWideBPE(nn.Module):

    def __init__(
        self,
        local_model,
        global_encoder,
        local_channels=32,
        global_channels=32,
    ):
        super().__init__()

        self.backbone = local_model.backbone
        self.global_encoder = global_encoder

        self.fusion = nn.Sequential(
            nn.Conv1d(
                local_channels + global_channels,
                local_channels,
                kernel_size=1,
            ),
            nn.GELU(),
        )

        self.profile_head = nn.Conv1d(
            local_channels,
            2,
            kernel_size=1,
        )

    def forward(
        self,
        x_local,
        global_token_ids,
        global_token_lengths,
        global_softmask,
        global_token_mask,
        local_bp_to_global_token,
    ):
        local_features = self.backbone(x_local)

        global_features = self.global_encoder(
            global_token_ids,
            global_token_lengths,
            global_softmask,
            global_token_mask,
        )

        gather_index = (
            local_bp_to_global_token
            .unsqueeze(1)
            .expand(
                -1,
                global_features.shape[1],
                -1,
            )
            .contiguous()
        )

        global_local_features = torch.gather(
            global_features,
            dim=2,
            index=gather_index,
        )

        fused = self.fusion(
            torch.cat(
                [
                    local_features,
                    global_local_features,
                ],
                dim=1,
            )
        )

        outputs = self.profile_head(fused)

        return (
            outputs[:, 0:1, :],
            outputs[:, 1:2, :],
        )


def create_wide_bpe_model():
    return CNNPresenceIntensityWideBPE(
        local_model=create_cnn_presence_intensity(),
        global_encoder=GlobalBPEEncoder(
            vocab_size=vocab_size,
            pad_token_id=PAD_TOKEN_ID,
            embedding_dim=16,
            channels=32,
            max_token_length=max(token_length.values()),
        ),
    )


print("Wide BPE model definitions ready")


Wide BPE model definitions ready


## Двухголовая модель

Ниже начинается обучение `presence + intensity` для новой архитектуры **LOCAL + Wide Global BPE**.

Первые две ячейки сделаны щадящими для DirectML:

1. первая только создаёт модель на GPU;
2. вторая делает минимальный smoke-test на **одном PLUS-окне**, без одновременного хранения PLUS+RC full-batch на GPU.


In [8]:
# 1. Создаём новую модель на DirectML — без forward в этой ячейке.

gc.collect()
torch.manual_seed(42)

multitask_model = create_wide_bpe_model().to(device)
multitask_model.eval()

parameter_count = sum(
    parameter.numel()
    for parameter in multitask_model.parameters()
    if parameter.requires_grad
)

print("Device:", device)
print("Parameters:", f"{parameter_count:,}")

assert parameter_count == 124_802


Device: privateuseone:0
Parameters: 124,802


In [11]:
# 2. DirectML smoke-test: один PLUS sample.
# Не переносим весь PLUS+RC batch на GPU одновременно.

smoke_tiles = (
    train_batch.tiles
    .iloc[:1]
    .reset_index(drop=True)
)

smoke_global = build_wide_global_bpe_batch(
    smoke_tiles,
    reverse_complement=False,
)

smoke_x = (
    train_batch.x_both_strands[:1]
    .contiguous()
    .to(device)
)

smoke_token_ids = (
    smoke_global["token_ids"]
    .contiguous()
    .to(device)
)

smoke_token_lengths = (
    smoke_global["token_lengths"]
    .contiguous()
    .to(device)
)

smoke_softmask = (
    smoke_global["softmask"]
    .contiguous()
    .to(device)
)

smoke_token_mask = (
    smoke_global["token_mask"]
    .contiguous()
    .to(device)
)

smoke_bp_to_token = (
    smoke_global["local_bp_to_token"]
    .contiguous()
    .to(device)
)

with torch.no_grad():
    smoke_presence_full, smoke_intensity_full = multitask_model(
        smoke_x,
        smoke_token_ids,
        smoke_token_lengths,
        smoke_softmask,
        smoke_token_mask,
        smoke_bp_to_token,
    )

    flank = profile_config.context_flank

    smoke_presence = smoke_presence_full[
        :, :, flank:-flank
    ]

    smoke_intensity = smoke_intensity_full[
        :, :, flank:-flank
    ]

expected_shape = (
    1,
    1,
    profile_config.target_length,
)

print("Presence:", smoke_presence.shape)
print("Intensity:", smoke_intensity.shape)
print("DirectML smoke-test: OK")

assert tuple(smoke_presence.shape) == expected_shape
assert tuple(smoke_intensity.shape) == expected_shape

# Не держим диагностические GPU tensors перед следующими ячейками.
del (
    smoke_x,
    smoke_token_ids,
    smoke_token_lengths,
    smoke_softmask,
    smoke_token_mask,
    smoke_bp_to_token,
    smoke_presence_full,
    smoke_intensity_full,
    smoke_presence,
    smoke_intensity,
)

gc.collect()
multitask_model.train()


Presence: torch.Size([1, 1, 1024])
Intensity: torch.Size([1, 1, 1024])
DirectML smoke-test: OK


CNNPresenceIntensityWideBPE(
  (backbone): CNNBackbone(
    (stem): Conv1d(5, 32, kernel_size=(1,), stride=(1,))
    (blocks): ModuleList(
      (0): ResidualBlock(
        (bn1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv1): Conv1d(32, 32, kernel_size=(7,), stride=(1,), padding=(3,))
        (bn2): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv2): Conv1d(32, 32, kernel_size=(7,), stride=(1,), padding=(3,))
      )
      (1): ResidualBlock(
        (bn1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv1): Conv1d(32, 32, kernel_size=(3,), stride=(1,), padding=(2,), dilation=(2,))
        (bn2): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv2): Conv1d(32, 32, kernel_size=(3,), stride=(1,), padding=(2,), dilation=(2,))
      )
      (2): ResidualBlock(
        (bn1): BatchNorm1d(32, eps=1e-05, momentu

In [12]:
# ============================================================
# 3. FULL BATCH PLUS + RC — DirectML
# ============================================================

multitask_model.eval()

B = len(train_batch.tiles)
flank = profile_config.context_flank


def forward_orientation_no_grad(
    model,
    x_cpu,
    global_batch,
):
    # --------------------------------------------------------
    # CPU -> DirectML
    # --------------------------------------------------------

    x_gpu = (
        x_cpu
        .contiguous()
        .to(device)
    )

    token_ids_gpu = (
        global_batch["token_ids"]
        .contiguous()
        .to(device)
    )

    token_lengths_gpu = (
        global_batch["token_lengths"]
        .contiguous()
        .to(device)
    )

    softmask_gpu = (
        global_batch["softmask"]
        .contiguous()
        .to(device)
    )

    token_mask_gpu = (
        global_batch["token_mask"]
        .contiguous()
        .to(device)
    )

    bp_to_token_gpu = (
        global_batch["local_bp_to_token"]
        .contiguous()
        .to(device)
    )

    # --------------------------------------------------------
    # Forward
    # --------------------------------------------------------

    with torch.no_grad():

        presence_full, intensity_full = model(
            x_gpu,
            token_ids_gpu,
            token_lengths_gpu,
            softmask_gpu,
            token_mask_gpu,
            bp_to_token_gpu,
        )

        presence = (
            presence_full[
                :, :, flank:-flank
            ]
            .detach()
            .cpu()
        )

        intensity = (
            intensity_full[
                :, :, flank:-flank
            ]
            .detach()
            .cpu()
        )

    # --------------------------------------------------------
    # Освобождаем DirectML tensors
    # --------------------------------------------------------

    del (
        x_gpu,
        token_ids_gpu,
        token_lengths_gpu,
        softmask_gpu,
        token_mask_gpu,
        bp_to_token_gpu,
        presence_full,
        intensity_full,
    )

    gc.collect()

    return presence, intensity


# ============================================================
# Строим wide BPE batches на CPU
# ============================================================

global_plus_batch = build_wide_global_bpe_batch(
    train_batch.tiles,
    reverse_complement=False,
)

global_rc_batch = build_wide_global_bpe_batch(
    train_batch.tiles,
    reverse_complement=True,
)


# ============================================================
# PLUS
# ============================================================

plus_presence, plus_intensity = (
    forward_orientation_no_grad(
        multitask_model,
        train_batch.x_both_strands[:B],
        global_plus_batch,
    )
)


# ============================================================
# RC
#
# Ничего НЕ flip'аем обратно.
# Это именно reverse-ориентация, соответствующая
# minus_target_reverse.
# ============================================================

minus_presence_reverse, minus_intensity_reverse = (
    forward_orientation_no_grad(
        multitask_model,
        train_batch.x_both_strands[B:],
        global_rc_batch,
    )
)


# ============================================================
# CHECK
# ============================================================

expected_shape = (
    B,
    1,
    profile_config.target_length,
)

print("Plus presence:         ", plus_presence.shape)
print("Minus presence reverse:", minus_presence_reverse.shape)

print("Plus intensity:        ", plus_intensity.shape)
print("Minus intensity reverse:", minus_intensity_reverse.shape)

print()

print(
    "Full PLUS + RC batch: OK"
)

assert tuple(plus_presence.shape) == expected_shape
assert tuple(minus_presence_reverse.shape) == expected_shape
assert tuple(plus_intensity.shape) == expected_shape
assert tuple(minus_intensity_reverse.shape) == expected_shape

multitask_model.train()

Plus presence:          torch.Size([4, 1, 1024])
Minus presence reverse: torch.Size([4, 1, 1024])
Plus intensity:         torch.Size([4, 1, 1024])
Minus intensity reverse: torch.Size([4, 1, 1024])

Full PLUS + RC batch: OK


CNNPresenceIntensityWideBPE(
  (backbone): CNNBackbone(
    (stem): Conv1d(5, 32, kernel_size=(1,), stride=(1,))
    (blocks): ModuleList(
      (0): ResidualBlock(
        (bn1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv1): Conv1d(32, 32, kernel_size=(7,), stride=(1,), padding=(3,))
        (bn2): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv2): Conv1d(32, 32, kernel_size=(7,), stride=(1,), padding=(3,))
      )
      (1): ResidualBlock(
        (bn1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv1): Conv1d(32, 32, kernel_size=(3,), stride=(1,), padding=(2,), dilation=(2,))
        (bn2): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv2): Conv1d(32, 32, kernel_size=(3,), stride=(1,), padding=(2,), dilation=(2,))
      )
      (2): ResidualBlock(
        (bn1): BatchNorm1d(32, eps=1e-05, momentu

In [13]:
# ============================================================
# 4. INTENSITY TARGETS
# ============================================================

plus_count = (
    train_batch.count[:, 0:1, :]
    .contiguous()
)

minus_count_reverse = (
    train_batch.count[:, 1:2, :]
    .flip(-1)
    .contiguous()
)

plus_intensity_target = torch.log1p(
    plus_count
)

minus_intensity_target_reverse = torch.log1p(
    minus_count_reverse
)

plus_intensity_mask = (
    plus_count > 0
)

minus_intensity_mask_reverse = (
    minus_count_reverse > 0
)


print("Plus count:", plus_count.shape)
print(
    "Minus count reverse:",
    minus_count_reverse.shape,
)

print(
    "Положительных plus:",
    int(plus_intensity_mask.sum()),
)

print(
    "Положительных minus:",
    int(minus_intensity_mask_reverse.sum()),
)


assert torch.equal(
    plus_intensity_mask.float(),
    train_batch.plus_target,
)

assert torch.equal(
    minus_intensity_mask_reverse.float(),
    train_batch.minus_target_reverse,
)

print("Intensity targets: OK")

Plus count: torch.Size([4, 1, 1024])
Minus count reverse: torch.Size([4, 1, 1024])
Положительных plus: 32
Положительных minus: 1
Intensity targets: OK


In [14]:
# ============================================================
# 5. REAL LOSS + BACKWARD — WIDE BPE MODEL
# ============================================================

import warnings

INTENSITY_LOSS_WEIGHT = 0.1

multitask_model.train()

optimizer = DirectMLAdam(
    multitask_model.parameters(),
    lr=1e-3,
)

B = len(train_batch.tiles)
flank = profile_config.context_flank


# ============================================================
# TARGETS -> DirectML
# ============================================================

plus_target_gpu = (
    train_batch.plus_target
    .contiguous()
    .to(device)
)

minus_target_reverse_gpu = (
    train_batch.minus_target_reverse
    .contiguous()
    .to(device)
)

plus_presence_weight_gpu = (
    train_batch.plus_loss_weight
    .contiguous()
    .to(device)
)

minus_presence_weight_reverse_gpu = (
    train_batch.minus_loss_weight_reverse
    .contiguous()
    .to(device)
)


plus_intensity_target_gpu = (
    plus_intensity_target
    .contiguous()
    .to(device)
)

minus_intensity_target_reverse_gpu = (
    minus_intensity_target_reverse
    .contiguous()
    .to(device)
)


plus_intensity_weight_gpu = (
    2.0
    * train_batch.plus_loss_weight
    * plus_intensity_mask.float()
).contiguous().to(device)

minus_intensity_weight_reverse_gpu = (
    2.0
    * train_batch.minus_loss_weight_reverse
    * minus_intensity_mask_reverse.float()
).contiguous().to(device)


# ============================================================
# FORWARD HELPER — С ГРАДИЕНТАМИ
# ============================================================

def forward_orientation_train(
    model,
    x_cpu,
    global_batch,
):
    x_gpu = (
        x_cpu
        .contiguous()
        .to(device)
    )

    token_ids_gpu = (
        global_batch["token_ids"]
        .contiguous()
        .to(device)
    )

    token_lengths_gpu = (
        global_batch["token_lengths"]
        .contiguous()
        .to(device)
    )

    softmask_gpu = (
        global_batch["softmask"]
        .contiguous()
        .to(device)
    )

    token_mask_gpu = (
        global_batch["token_mask"]
        .contiguous()
        .to(device)
    )

    bp_to_token_gpu = (
        global_batch["local_bp_to_token"]
        .contiguous()
        .to(device)
    )

    presence_full, intensity_full = model(
        x_gpu,
        token_ids_gpu,
        token_lengths_gpu,
        softmask_gpu,
        token_mask_gpu,
        bp_to_token_gpu,
    )

    presence = presence_full[
        :, :, flank:-flank
    ]

    intensity = intensity_full[
        :, :, flank:-flank
    ]

    return presence, intensity


# ============================================================
# FORWARD PLUS + RC
# ============================================================

optimizer.zero_grad(set_to_none=True)

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")

    plus_presence, plus_intensity = (
        forward_orientation_train(
            multitask_model,
            train_batch.x_both_strands[:B],
            global_plus_batch,
        )
    )

    minus_presence_reverse, minus_intensity_reverse = (
        forward_orientation_train(
            multitask_model,
            train_batch.x_both_strands[B:],
            global_rc_batch,
        )
    )


    # ========================================================
    # PRESENCE LOSS
    # ========================================================

    presence_loss = (
        dml_weighted_profile_bce_with_logits(
            plus_presence,
            minus_presence_reverse,
            plus_target_gpu,
            minus_target_reverse_gpu,
            plus_presence_weight_gpu,
            minus_presence_weight_reverse_gpu,
        )
    )


    # ========================================================
    # INTENSITY LOSS
    # ========================================================

    intensity_loss = (
        dml_weighted_huber(
            plus_intensity,
            plus_intensity_target_gpu,
            plus_intensity_weight_gpu,
        )
        +
        dml_weighted_huber(
            minus_intensity_reverse,
            minus_intensity_target_reverse_gpu,
            minus_intensity_weight_reverse_gpu,
        )
    )


    # ========================================================
    # TOTAL LOSS
    # ========================================================

    total_loss = (
        presence_loss
        + INTENSITY_LOSS_WEIGHT * intensity_loss
    )

    total_loss.backward()


# ============================================================
# GRADIENT CHECK
# ============================================================

def module_grad_norm(module):
    total = 0.0

    for parameter in module.parameters():
        if parameter.grad is not None:
            total += (
                parameter.grad.detach().float()
                .square().sum().item()
            )

    return total ** 0.5


local_grad = module_grad_norm(
    multitask_model.backbone
)

global_grad = module_grad_norm(
    multitask_model.global_encoder
)

fusion_grad = module_grad_norm(
    multitask_model.fusion
)

head_grad = module_grad_norm(
    multitask_model.profile_head
)


# ============================================================
# UPDATE WEIGHTS
# ============================================================

optimizer.step()


fallback_warnings = [
    str(item.message)
    for item in caught
    if "fall back" in str(item.message).lower()
]


print("=" * 80)
print("REAL WIDE-BPE TRAINING STEP")
print("=" * 80)

print("Presence loss:", presence_loss.item())
print("Intensity loss:", intensity_loss.item())
print("Total loss:", total_loss.item())

print()

print("LOCAL grad: ", local_grad)
print("GLOBAL grad:", global_grad)
print("Fusion grad:", fusion_grad)
print("Head grad:  ", head_grad)

print()

print("CPU fallback:", fallback_warnings)


assert local_grad > 0
assert global_grad > 0
assert fusion_grad > 0
assert head_grad > 0
assert not fallback_warnings

print("Real backward + optimizer step: OK")

REAL WIDE-BPE TRAINING STEP
Presence loss: 0.697140634059906
Intensity loss: 0.9845865964889526
Total loss: 0.7955992817878723

LOCAL grad:  0.43045861445282185
GLOBAL grad: 0.0035990709631580665
Fusion grad: 0.15265628750425062
Head grad:   0.16978881548075145

CPU fallback: []
Real backward + optimizer step: OK


In [15]:
# ============================================================
# 6. 200-STEP PILOT — LOCAL + WIDE GLOBAL BPE
# ============================================================

import math
import time
import warnings


PILOT_STEPS = 200
PILOT_BATCH_SIZE = 4
PILOT_SEED = 42
LOG_EVERY = 20
INTENSITY_LOSS_WEIGHT = 0.1

torch.manual_seed(PILOT_SEED)

pilot_generator = ProfileWindowGenerator(
    prepared_split=split,
    sequences=sequences,
    part="train",
    config=profile_config,
    seed=PILOT_SEED,
)

# ВАЖНО: новая модель с нуля
pilot_model = (
    create_wide_bpe_model()
    .to(device)
)

pilot_optimizer = DirectMLAdam(
    pilot_model.parameters(),
    lr=1e-3,
)

pilot_model.train()

pilot_history = []

ema_presence = None
ema_intensity = None
ema_total = None

seen_position_strand = 0
started_at = time.perf_counter()


with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")

    for step in range(1, PILOT_STEPS + 1):

        # ====================================================
        # НОВЫЙ RANDOM BATCH
        # ====================================================

        batch = pilot_generator.sample_batch(
            batch_size=PILOT_BATCH_SIZE
        )

        B = len(batch.tiles)

        # ====================================================
        # WIDE BPE — строим заново для текущих tiles
        # ====================================================

        global_plus = build_wide_global_bpe_batch(
            batch.tiles,
            reverse_complement=False,
        )

        global_rc = build_wide_global_bpe_batch(
            batch.tiles,
            reverse_complement=True,
        )

        # ====================================================
        # TARGETS
        # ====================================================

        plus_target_gpu = (
            batch.plus_target
            .contiguous()
            .to(device)
        )

        minus_target_reverse_gpu = (
            batch.minus_target_reverse
            .contiguous()
            .to(device)
        )

        plus_presence_weight_gpu = (
            batch.plus_loss_weight
            .contiguous()
            .to(device)
        )

        minus_presence_weight_reverse_gpu = (
            batch.minus_loss_weight_reverse
            .contiguous()
            .to(device)
        )

        # ----------------------------------------------------
        # Intensity
        # ----------------------------------------------------

        plus_count = (
            batch.count[:, 0:1, :]
            .contiguous()
        )

        minus_count_reverse = (
            batch.count[:, 1:2, :]
            .flip(-1)
            .contiguous()
        )

        plus_intensity_target_gpu = (
            torch.log1p(plus_count)
            .contiguous()
            .to(device)
        )

        minus_intensity_target_reverse_gpu = (
            torch.log1p(minus_count_reverse)
            .contiguous()
            .to(device)
        )

        plus_intensity_weight_gpu = (
            2.0
            * batch.plus_loss_weight
            * (plus_count > 0).float()
        ).contiguous().to(device)

        minus_intensity_weight_reverse_gpu = (
            2.0
            * batch.minus_loss_weight_reverse
            * (minus_count_reverse > 0).float()
        ).contiguous().to(device)

        # ====================================================
        # ZERO GRAD
        # ====================================================

        pilot_optimizer.zero_grad(
            set_to_none=True
        )

        # ====================================================
        # PLUS FORWARD
        # ====================================================

        plus_presence, plus_intensity = (
            forward_orientation_train(
                pilot_model,
                batch.x_both_strands[:B],
                global_plus,
            )
        )

        # ====================================================
        # RC FORWARD
        # ====================================================

        (
            minus_presence_reverse,
            minus_intensity_reverse,
        ) = forward_orientation_train(
            pilot_model,
            batch.x_both_strands[B:],
            global_rc,
        )

        # ====================================================
        # LOSS
        # ====================================================

        presence_loss = (
            dml_weighted_profile_bce_with_logits(
                plus_presence,
                minus_presence_reverse,
                plus_target_gpu,
                minus_target_reverse_gpu,
                plus_presence_weight_gpu,
                minus_presence_weight_reverse_gpu,
            )
        )

        intensity_loss = (
            dml_weighted_huber(
                plus_intensity,
                plus_intensity_target_gpu,
                plus_intensity_weight_gpu,
            )
            +
            dml_weighted_huber(
                minus_intensity_reverse,
                minus_intensity_target_reverse_gpu,
                minus_intensity_weight_reverse_gpu,
            )
        )

        total_loss = (
            presence_loss
            + INTENSITY_LOSS_WEIGHT
            * intensity_loss
        )

        # ====================================================
        # SAFETY
        # ====================================================

        total_value = total_loss.item()

        if not math.isfinite(total_value):
            raise FloatingPointError(
                f"Non-finite loss at step {step}"
            )

        # ====================================================
        # BACKWARD + UPDATE
        # ====================================================

        total_loss.backward()
        pilot_optimizer.step()

        # ====================================================
        # LOGGING
        # ====================================================

        presence_value = presence_loss.item()
        intensity_value = intensity_loss.item()

        if ema_total is None:
            ema_presence = presence_value
            ema_intensity = intensity_value
            ema_total = total_value

        else:
            ema_presence = (
                0.95 * ema_presence
                + 0.05 * presence_value
            )

            ema_intensity = (
                0.95 * ema_intensity
                + 0.05 * intensity_value
            )

            ema_total = (
                0.95 * ema_total
                + 0.05 * total_value
            )

        seen_position_strand += int(
            batch.mask.sum()
        )

        pilot_history.append(
            {
                "step": step,
                "presence_loss": presence_value,
                "intensity_loss": intensity_value,
                "total_loss": total_value,
                "ema_presence": ema_presence,
                "ema_intensity": ema_intensity,
                "ema_total": ema_total,
            }
        )

        if (
            step == 1
            or step % LOG_EVERY == 0
        ):
            elapsed = (
                time.perf_counter()
                - started_at
            )

            print(
                f"step={step:3d} "
                f"presence={presence_value:.4f} "
                f"intensity={intensity_value:.4f} "
                f"total={total_value:.4f} "
                f"ema=({ema_presence:.4f}, "
                f"{ema_intensity:.4f}, "
                f"{ema_total:.4f}) "
                f"positions/s="
                f"{seen_position_strand / elapsed:,.0f}"
            )


fallback_warnings = [
    str(item.message)
    for item in caught
    if "fall back" in str(item.message).lower()
]

elapsed = (
    time.perf_counter()
    - started_at
)

print()
print("Время:", round(elapsed, 2), "сек")
print(
    "Финальный EMA presence:",
    round(ema_presence, 6),
)
print(
    "Финальный EMA intensity:",
    round(ema_intensity, 6),
)
print(
    "Финальный EMA total:",
    round(ema_total, 6),
)
print(
    "CPU fallback:",
    fallback_warnings,
)

assert not fallback_warnings

step=  1 presence=0.6971 intensity=0.9846 total=0.7956 ema=(0.6971, 0.9846, 0.7956) positions/s=33,973
step= 20 presence=0.6550 intensity=0.4767 total=0.7027 ema=(0.6852, 0.6037, 0.7456) positions/s=31,172
step= 40 presence=0.5807 intensity=0.4626 total=0.6270 ema=(0.6549, 0.4505, 0.7000) positions/s=31,969
step= 60 presence=0.5491 intensity=0.3203 total=0.5811 ema=(0.6172, 0.4072, 0.6579) positions/s=31,423
step= 80 presence=0.6068 intensity=0.3443 total=0.6412 ema=(0.5754, 0.3034, 0.6057) positions/s=31,391
step=100 presence=0.6689 intensity=1.2974 total=0.7986 ema=(0.5757, 0.3507, 0.6108) positions/s=31,889
step=120 presence=0.4104 intensity=0.1226 total=0.4226 ema=(0.5475, 0.3011, 0.5776) positions/s=32,119
step=140 presence=0.6220 intensity=0.7599 total=0.6980 ema=(0.5547, 0.2597, 0.5807) positions/s=32,319
step=160 presence=0.4084 intensity=0.1248 total=0.4209 ema=(0.5471, 0.2557, 0.5727) positions/s=32,570
step=180 presence=0.4253 intensity=0.1855 total=0.4438 ema=(0.4946, 0.194

In [16]:
# ============================================================
# 7. CONTINUE PILOT -> 10 000 STEPS
# ============================================================

TOTAL_STEPS = 10_000
START_STEP = PILOT_STEPS          # 200
LOG_EVERY = 250

CHECKPOINT_STEPS = {
    2_000,
    5_000,
    10_000,
}

RUN_DIR = (
    PROJECT_ROOT
    / "artifacts"
    / "CNN_EXP006_WIDE_BPE"
    / "run_001"
)

RUN_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

pilot_model.train()

started_at = time.perf_counter()

seen_position_strand = 0


for step in range(
    START_STEP + 1,
    TOTAL_STEPS + 1,
):

    # ========================================================
    # RANDOM TRAINING BATCH
    # ========================================================

    batch = pilot_generator.sample_batch(
        batch_size=PILOT_BATCH_SIZE
    )

    B = len(batch.tiles)

    # ========================================================
    # BUILD WIDE BPE INPUTS
    # ========================================================

    global_plus = build_wide_global_bpe_batch(
        batch.tiles,
        reverse_complement=False,
    )

    global_rc = build_wide_global_bpe_batch(
        batch.tiles,
        reverse_complement=True,
    )

    # ========================================================
    # TARGETS
    # ========================================================

    plus_target_gpu = (
        batch.plus_target
        .contiguous()
        .to(device)
    )

    minus_target_reverse_gpu = (
        batch.minus_target_reverse
        .contiguous()
        .to(device)
    )

    plus_presence_weight_gpu = (
        batch.plus_loss_weight
        .contiguous()
        .to(device)
    )

    minus_presence_weight_reverse_gpu = (
        batch.minus_loss_weight_reverse
        .contiguous()
        .to(device)
    )

    # ========================================================
    # INTENSITY TARGETS
    # ========================================================

    plus_count = (
        batch.count[:, 0:1, :]
        .contiguous()
    )

    minus_count_reverse = (
        batch.count[:, 1:2, :]
        .flip(-1)
        .contiguous()
    )

    plus_intensity_target_gpu = (
        torch.log1p(
            plus_count
        )
        .contiguous()
        .to(device)
    )

    minus_intensity_target_reverse_gpu = (
        torch.log1p(
            minus_count_reverse
        )
        .contiguous()
        .to(device)
    )

    plus_intensity_weight_gpu = (
        2.0
        * batch.plus_loss_weight
        * (plus_count > 0).float()
    ).contiguous().to(device)

    minus_intensity_weight_reverse_gpu = (
        2.0
        * batch.minus_loss_weight_reverse
        * (minus_count_reverse > 0).float()
    ).contiguous().to(device)

    # ========================================================
    # FORWARD
    # ========================================================

    pilot_optimizer.zero_grad(
        set_to_none=True
    )

    plus_presence, plus_intensity = (
        forward_orientation_train(
            pilot_model,
            batch.x_both_strands[:B],
            global_plus,
        )
    )

    (
        minus_presence_reverse,
        minus_intensity_reverse,
    ) = forward_orientation_train(
        pilot_model,
        batch.x_both_strands[B:],
        global_rc,
    )

    # ========================================================
    # LOSS
    # ========================================================

    presence_loss = (
        dml_weighted_profile_bce_with_logits(
            plus_presence,
            minus_presence_reverse,
            plus_target_gpu,
            minus_target_reverse_gpu,
            plus_presence_weight_gpu,
            minus_presence_weight_reverse_gpu,
        )
    )

    intensity_loss = (
        dml_weighted_huber(
            plus_intensity,
            plus_intensity_target_gpu,
            plus_intensity_weight_gpu,
        )
        +
        dml_weighted_huber(
            minus_intensity_reverse,
            minus_intensity_target_reverse_gpu,
            minus_intensity_weight_reverse_gpu,
        )
    )

    total_loss = (
        presence_loss
        + INTENSITY_LOSS_WEIGHT
        * intensity_loss
    )

    # ========================================================
    # FINITE CHECK
    # ========================================================

    total_value = total_loss.item()

    if not math.isfinite(total_value):
        raise FloatingPointError(
            f"Non-finite loss at step {step}"
        )

    # ========================================================
    # BACKWARD
    # ========================================================

    total_loss.backward()

    pilot_optimizer.step()

    # ========================================================
    # EMA
    # ========================================================

    presence_value = presence_loss.item()
    intensity_value = intensity_loss.item()

    ema_presence = (
        0.95 * ema_presence
        + 0.05 * presence_value
    )

    ema_intensity = (
        0.95 * ema_intensity
        + 0.05 * intensity_value
    )

    ema_total = (
        0.95 * ema_total
        + 0.05 * total_value
    )

    pilot_history.append(
        {
            "step": step,
            "presence_loss": presence_value,
            "intensity_loss": intensity_value,
            "total_loss": total_value,
            "ema_presence": ema_presence,
            "ema_intensity": ema_intensity,
            "ema_total": ema_total,
        }
    )

    seen_position_strand += int(
        batch.mask.sum()
    )

    # ========================================================
    # LOG
    # ========================================================

    if step % LOG_EVERY == 0:

        elapsed = (
            time.perf_counter()
            - started_at
        )

        steps_done = (
            step
            - START_STEP
        )

        sec_per_step = (
            elapsed
            / steps_done
        )

        eta_seconds = (
            TOTAL_STEPS
            - step
        ) * sec_per_step

        print(
            f"step={step:5d} "
            f"presence={presence_value:.4f} "
            f"intensity={intensity_value:.4f} "
            f"total={total_value:.4f} "
            f"ema=({ema_presence:.4f}, "
            f"{ema_intensity:.4f}, "
            f"{ema_total:.4f}) "
            f"positions/s="
            f"{seen_position_strand / elapsed:,.0f} "
            f"ETA={eta_seconds / 60:.1f} min"
        )

    # ========================================================
    # CHECKPOINT
    # ========================================================

    if step in CHECKPOINT_STEPS:

        checkpoint_path = (
            RUN_DIR
            / f"checkpoint_step_{step:05d}.pt"
        )

        model_state_cpu = {
            name: tensor.detach().cpu()
            for name, tensor
            in pilot_model.state_dict().items()
        }

        torch.save(
            {
                "step": step,
                "model_state_dict":
                    model_state_cpu,
                "ema_presence":
                    ema_presence,
                "ema_intensity":
                    ema_intensity,
                "ema_total":
                    ema_total,
                "config": {
                    "global_length":
                        GLOBAL_LENGTH,
                    "global_flank":
                        GLOBAL_FLANK,
                    "bpe_vocab_size":
                        vocab_size,
                    "bpe_merges":
                        len(merge_history),
                    "intensity_loss_weight":
                        INTENSITY_LOSS_WEIGHT,
                },
            },
            checkpoint_path,
        )

        print(
            "Checkpoint:",
            checkpoint_path,
        )


# ============================================================
# FINAL
# ============================================================

elapsed = (
    time.perf_counter()
    - started_at
)

print()
print("=" * 80)
print("10 000 STEPS COMPLETE")
print("=" * 80)

print(
    "Дополнительное время:",
    round(elapsed / 60, 2),
    "мин",
)

print(
    "Final EMA presence:",
    round(ema_presence, 6),
)

print(
    "Final EMA intensity:",
    round(ema_intensity, 6),
)

print(
    "Final EMA total:",
    round(ema_total, 6),
)

step=  250 presence=0.4911 intensity=0.4811 total=0.5393 ema=(0.5146, 0.3374, 0.5483) positions/s=33,829 ETA=36.7 min
step=  500 presence=0.5226 intensity=0.0315 total=0.5257 ema=(0.4905, 0.2301, 0.5135) positions/s=33,297 ETA=36.6 min
step=  750 presence=0.6279 intensity=0.2571 total=0.6536 ema=(0.5005, 0.2476, 0.5253) positions/s=32,579 ETA=36.2 min
step= 1000 presence=0.2892 intensity=0.1934 total=0.3085 ema=(0.4621, 0.2490, 0.4870) positions/s=32,480 ETA=35.2 min
step= 1250 presence=0.2533 intensity=0.8742 total=0.3407 ema=(0.4379, 0.2379, 0.4617) positions/s=32,337 ETA=34.4 min
step= 1500 presence=0.3517 intensity=0.1382 total=0.3655 ema=(0.4493, 0.2018, 0.4695) positions/s=32,233 ETA=33.5 min
step= 1750 presence=0.5747 intensity=0.0837 total=0.5830 ema=(0.4233, 0.2569, 0.4490) positions/s=32,291 ETA=32.5 min
step= 2000 presence=0.2780 intensity=0.3444 total=0.3124 ema=(0.4031, 0.2760, 0.4307) positions/s=32,335 ETA=31.4 min
Checkpoint: D:\Projects\EPIC\EPIC\artifacts\CNN_EXP006_W

In [17]:
# ============================================================
# 8. LOAD 10K WIDE-BPE CHECKPOINT + VALIDATION SETUP
# ============================================================

WIDE_BPE_CHECKPOINT_PATH = (
    PROJECT_ROOT
    / "artifacts"
    / "CNN_EXP006_WIDE_BPE"
    / "run_001"
    / "checkpoint_step_10000.pt"
)

checkpoint = torch.load(
    WIDE_BPE_CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False,
)

validation_model = (
    create_wide_bpe_model()
    .to(device)
)

validation_model.load_state_dict(
    checkpoint["model_state_dict"]
)

validation_model.eval()


# ============================================================
# VALIDATION GENERATOR
# ============================================================

validation_generator = ProfileWindowGenerator(
    prepared_split=split,
    sequences=sequences,
    part="validation",
    config=profile_config,
    seed=42,
)


# ============================================================
# CHECK
# ============================================================

parameter_count = sum(
    parameter.numel()
    for parameter in validation_model.parameters()
    if parameter.requires_grad
)

print("=" * 80)
print("WIDE-BPE-50 — VALIDATION SETUP")
print("=" * 80)

print(
    "Checkpoint:",
    WIDE_BPE_CHECKPOINT_PATH,
)

print(
    "Checkpoint step:",
    checkpoint["step"],
)

print(
    "Parameters:",
    f"{parameter_count:,}",
)

print(
    "Validation tiles:",
    len(validation_generator.tile_index),
)

print(
    "Validation position-strand:",
    int(
        validation_generator
        .tile_index["allowed_position_strand"]
        .sum()
    ),
)

print(
    "Validation positives:",
    validation_generator.total_positive,
)

print(
    "Device:",
    next(validation_model.parameters()).device,
)

assert checkpoint["step"] == 10_000
assert parameter_count == 124_802

print()
print("Checkpoint + validation setup: OK")

WIDE-BPE-50 — VALIDATION SETUP
Checkpoint: D:\Projects\EPIC\EPIC\artifacts\CNN_EXP006_WIDE_BPE\run_001\checkpoint_step_10000.pt
Checkpoint step: 10000
Parameters: 124,802
Validation tiles: 47743
Validation position-strand: 83807486
Validation positives: 56205
Device: privateuseone:0

Checkpoint + validation setup: OK


In [18]:
import inspect
from ml_project.epic_cnn.evaluation import (
    evaluate_presence_intensity_model,
)

print(
    inspect.getsource(
        evaluate_presence_intensity_model
    )
)

def evaluate_presence_intensity_model(
    model: torch.nn.Module,
    generator: ProfileWindowGenerator,
    *,
    device: Any,
    batch_size: int = 8,
    score_decimals: int = 5,
    progress_every_batches: int = 500,
    progress: Callable[[str], None] | None = print,
    fail_on_cpu_fallback: bool = True,
) -> ProfileEvaluationResult:
    """Evaluate presence, intensity and their fixed expected-signal combination."""

    if not isinstance(batch_size, int) or batch_size <= 0:
        raise ValueError("batch_size must be a positive integer")
    if not isinstance(score_decimals, int) or not 1 <= score_decimals <= 6:
        raise ValueError("score_decimals must be an integer from 1 to 6")
    if not isinstance(progress_every_batches, int) or progress_every_batches <= 0:
        raise ValueError("progress_every_batches must be positive")

    score_names = (
        "presence_probability",
        "intensity_prediction",
        "expected_signal",
    )
    tiles = generator.tile_

In [19]:
# ============================================================
# 9. FULL VALIDATION — WIDE BPE
# ============================================================

import time
import warnings
import numpy as np
import pandas as pd

from scipy.special import expit
import ml_project.epic_cnn.evaluation as epic_eval


EVAL_BATCH_SIZE = 4
SCORE_DECIMALS = 5
PROGRESS_EVERY = 500


def evaluate_wide_bpe_presence_intensity(
    model,
    generator,
    *,
    device,
    batch_size=4,
    score_decimals=5,
    progress_every_batches=500,
):

    score_names = (
        "presence_probability",
        "intensity_prediction",
        "expected_signal",
    )

    tiles = generator.tile_index

    contigs = tuple(
        str(value)
        for value in tiles["contig"].unique()
    )

    score_scale = 10 ** score_decimals
    score_levels = score_scale + 1

    accumulators = {
        score_name: {
            contig: epic_eval._MetricAccumulator(
                score_levels
            )
            for contig in contigs
        }
        for score_name in score_names
    }

    processed_positions = 0
    processed_positives = 0

    started_at = time.perf_counter()

    was_training = model.training
    model.eval()

    try:

        with warnings.catch_warnings(
            record=True
        ) as caught:

            warnings.simplefilter("always")

            for batch_number, start in enumerate(
                range(
                    0,
                    len(tiles),
                    batch_size,
                ),
                start=1,
            ):

                # ============================================
                # TILE BATCH
                # ============================================

                selected = tiles.iloc[
                    start:start + batch_size
                ]

                batch = generator.make_batch(
                    selected
                )

                current_batch_size = len(
                    selected
                )

                # ============================================
                # WIDE BPE INPUTS
                # ============================================

                global_plus = (
                    build_wide_global_bpe_batch(
                        batch.tiles,
                        reverse_complement=False,
                    )
                )

                global_rc = (
                    build_wide_global_bpe_batch(
                        batch.tiles,
                        reverse_complement=True,
                    )
                )

                # ============================================
                # FORWARD PLUS
                # ============================================

                plus_presence, plus_intensity = (
                    forward_orientation_no_grad(
                        model,
                        batch.x_both_strands[
                            :current_batch_size
                        ],
                        global_plus,
                    )
                )

                # ============================================
                # FORWARD RC
                #
                # Здесь результат остаётся в RC orientation.
                # Ниже ::-1 вернёт minus в genomic orientation,
                # точно как в старом evaluator.
                # ============================================

                (
                    minus_presence_reverse,
                    minus_intensity_reverse,
                ) = forward_orientation_no_grad(
                    model,
                    batch.x_both_strands[
                        current_batch_size:
                    ],
                    global_rc,
                )

                # ============================================
                # SAME SCORE CONSTRUCTION AS ORIGINAL
                # ============================================

                presence_logits = np.stack(
                    (
                        plus_presence.numpy()[
                            :, 0, :
                        ],

                        minus_presence_reverse.numpy()[
                            :, 0, ::-1
                        ],
                    ),
                    axis=1,
                )

                intensity_raw = np.stack(
                    (
                        plus_intensity.numpy()[
                            :, 0, :
                        ],

                        minus_intensity_reverse.numpy()[
                            :, 0, ::-1
                        ],
                    ),
                    axis=1,
                )

                presence_probability = expit(
                    presence_logits
                )

                intensity_score = expit(
                    intensity_raw
                )

                expected_raw = (
                    presence_probability
                    * np.expm1(
                        np.clip(
                            intensity_raw,
                            0.0,
                            20.0,
                        )
                    )
                )

                expected_score = (
                    expected_raw
                    / (1.0 + expected_raw)
                )

                scores = {
                    "presence_probability":
                        presence_probability,

                    "intensity_prediction":
                        intensity_score,

                    "expected_signal":
                        expected_score,
                }

                # ============================================
                # TARGETS
                # ============================================

                mask = batch.mask.numpy()
                counts = batch.count.numpy()

                selected_contigs = (
                    selected["contig"]
                    .astype(str)
                    .to_numpy()
                )

                # ============================================
                # ACCUMULATE METRICS
                # ============================================

                for score_name, score in (
                    scores.items()
                ):

                    quantized = np.rint(
                        np.clip(
                            score,
                            0.0,
                            1.0,
                        )
                        * score_scale
                    ).astype(np.int32)

                    for contig in np.unique(
                        selected_contigs
                    ):

                        rows = np.flatnonzero(
                            selected_contigs
                            == contig
                        )

                        accumulators[
                            score_name
                        ][contig].add(
                            score[rows],
                            quantized[rows],
                            mask[rows],
                            counts[rows],
                        )

                # ============================================
                # COUNTERS
                # ============================================

                processed_positions += int(
                    mask.sum()
                )

                processed_positives += int(
                    (
                        mask
                        & (counts > 0)
                    ).sum()
                )

                finished = (
                    start
                    + current_batch_size
                    >= len(tiles)
                )

                if (
                    batch_number == 1
                    or batch_number
                    % progress_every_batches
                    == 0
                    or finished
                ):

                    elapsed = (
                        time.perf_counter()
                        - started_at
                    )

                    print(
                        f"tiles="
                        f"{start + current_batch_size:>6}"
                        f"/{len(tiles)} "
                        f"positions="
                        f"{processed_positions:>9} "
                        f"positions/s="
                        f"{processed_positions / elapsed:,.0f}"
                    )

        fallback_warnings = tuple(
            str(item.message)
            for item in caught
            if "fall back"
            in str(item.message).lower()
        )

    finally:

        model.train(
            was_training
        )

    # ========================================================
    # CONSISTENCY CHECK
    # ========================================================

    expected_positions = int(
        tiles[
            "allowed_position_strand"
        ].sum()
    )

    expected_positives = int(
        tiles[
            "positive_positions"
        ].sum()
    )

    assert (
        processed_positions
        == expected_positions
    )

    assert (
        processed_positives
        == expected_positives
    )

    if fallback_warnings:
        raise RuntimeError(
            "DirectML CPU fallback detected: "
            + " | ".join(
                fallback_warnings
            )
        )

    # ========================================================
    # FINAL METRICS
    # ========================================================

    rows = []

    for score_name in score_names:

        by_contig = accumulators[
            score_name
        ]

        overall = (
            epic_eval._MetricAccumulator(
                score_levels
            )
        )

        overall.merge(
            list(
                by_contig.values()
            )
        )

        overall_row = (
            overall.metrics(
                "overall"
            )
        )

        overall_row["score"] = (
            score_name
        )

        rows.append(
            overall_row
        )

        for contig in contigs:

            row = (
                by_contig[
                    contig
                ].metrics(
                    contig
                )
            )

            row["score"] = (
                score_name
            )

            rows.append(
                row
            )

    metrics = pd.DataFrame(
        rows
    )[
        [
            "score",
            "segment",
            "position_strand",
            "positives",
            "prevalence",
            "average_precision",
            "ap_lift",
            "epic_spearman",
            "quantized_spearman",
        ]
    ]

    elapsed_seconds = (
        time.perf_counter()
        - started_at
    )

    return (
        metrics,
        elapsed_seconds,
        processed_positions
        / elapsed_seconds,
    )


# ============================================================
# RUN FULL VALIDATION
# ============================================================

wide_bpe_metrics, wide_bpe_eval_seconds, wide_bpe_eval_speed = (
    evaluate_wide_bpe_presence_intensity(
        validation_model,
        validation_generator,
        device=device,
        batch_size=EVAL_BATCH_SIZE,
        score_decimals=SCORE_DECIMALS,
        progress_every_batches=PROGRESS_EVERY,
    )
)


# ============================================================
# SHOW OVERALL RESULT
# ============================================================

wide_bpe_overall = (
    wide_bpe_metrics[
        wide_bpe_metrics[
            "segment"
        ] == "overall"
    ]
    .reset_index(drop=True)
)

print()
print("=" * 80)
print("WIDE-BPE-50 — HOLDOUT RESULT")
print("=" * 80)

display(
    wide_bpe_overall
)

print(
    "Evaluation time:",
    round(
        wide_bpe_eval_seconds / 60,
        2,
    ),
    "min",
)

print(
    "Positions/s:",
    f"{wide_bpe_eval_speed:,.0f}",
)

tiles=     4/47743 positions=     5414 positions/s=19,732
tiles=  2000/47743 positions=  3270676 positions/s=24,101
tiles=  4000/47743 positions=  6861742 positions/s=25,096
tiles=  6000/47743 positions= 10486418 positions/s=25,579
tiles=  8000/47743 positions= 14166170 positions/s=25,878
tiles= 10000/47743 positions= 17785210 positions/s=26,046
tiles= 12000/47743 positions= 21458144 positions/s=26,173
tiles= 14000/47743 positions= 25044318 positions/s=26,178
tiles= 16000/47743 positions= 28627148 positions/s=26,241
tiles= 18000/47743 positions= 32126086 positions/s=26,197
tiles= 20000/47743 positions= 35584246 positions/s=26,021
tiles= 22000/47743 positions= 39054518 positions/s=25,914
tiles= 24000/47743 positions= 42571942 positions/s=25,903
tiles= 26000/47743 positions= 46125962 positions/s=25,958
tiles= 28000/47743 positions= 49714988 positions/s=25,966
tiles= 30000/47743 positions= 53186290 positions/s=25,954
tiles= 32000/47743 positions= 56582858 positions/s=25,892
tiles= 34000/4

,score,segment,position_strand,positives,prevalence,average_precision,ap_lift,epic_spearman,quantized_spearman
0,presence_probability,overall,83807486,56205,0.000671,0.031211,46.538733,0.176440,0.157779
1,intensity_prediction,overall,83807486,56205,0.000671,0.004325,6.448326,0.184515,0.208483
2,expected_signal,overall,83807486,56205,0.000671,0.030879,46.043724,0.185370,0.182022


Evaluation time: 54.54 min
Positions/s: 25,610


In [ ]:
import copy
import math
import time
import warnings

import pandas as pd


def model_state_on_cpu(model):
    return {
        name: tensor.detach().cpu().clone()
        for name, tensor in model.state_dict().items()
    }


def tree_to_cpu(value):
    if torch.is_tensor(value):
        return value.detach().cpu().clone()

    if isinstance(value, dict):
        return {
            key: tree_to_cpu(item)
            for key, item in value.items()
        }

    if isinstance(value, list):
        return [
            tree_to_cpu(item)
            for item in value
        ]

    if isinstance(value, tuple):
        return tuple(
            tree_to_cpu(item)
            for item in value
        )

    return value


def save_exp002_checkpoint(
    path,
    *,
    step,
    elapsed_seconds,
    seen_position_strand,
    history,
):
    checkpoint = {
        "experiment": "CNN-EXP-006",
        "model_type": "presence_intensity",
        "step": step,
        "model_state_dict": model_state_on_cpu(
            exp002_model
        ),
        "optimizer_state_dict": tree_to_cpu(
            exp002_optimizer.state_dict()
        ),
        "generator_rng_state": copy.deepcopy(
            exp002_generator.rng.bit_generator.state
        ),
        "torch_rng_state": torch.get_rng_state(),
        "profile_config": {
            "target_length": (
                profile_config.target_length
            ),
            "context_flank": (
                profile_config.context_flank
            ),
            "positive_branch_fraction": (
                profile_config
                .positive_branch_fraction
            ),
        },
        "training_config": {
            "seed": EXP006_SEED,
            "steps": EXP006_STEPS,
            "batch_size": EXP006_BATCH_SIZE,
            "learning_rate": 1e-3,
            "intensity_loss_weight": (
                INTENSITY_LOSS_WEIGHT
            ),
            "huber_delta": 1.0,
        },
        "seen_position_strand": (
            seen_position_strand
        ),
        "elapsed_seconds": elapsed_seconds,
        "history": list(history),
    }

    torch.save(checkpoint, path)


exp002_model.train()

exp002_history = []
ema_presence = None
ema_intensity = None
ema_total = None
seen_position_strand = 0
started_at = time.perf_counter()

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")

    for step in range(1, EXP006_STEPS + 1):
        batch = exp002_generator.sample_batch(
            batch_size=EXP006_BATCH_SIZE
        )

        x_gpu = batch.x_both_strands.to(device)

        plus_target_gpu = (
            batch.plus_target.to(device)
        )
        minus_target_reverse_gpu = (
            batch.minus_target_reverse.to(device)
        )

        plus_presence_weight_gpu = (
            batch.plus_loss_weight.to(device)
        )
        minus_presence_weight_reverse_gpu = (
            batch.minus_loss_weight_reverse.to(device)
        )

        # Всё выравнивание minus выполняется на CPU.
        plus_count = (
            batch.count[:, 0:1, :]
            .contiguous()
        )
        minus_count_reverse = (
            batch.count[:, 1:2, :]
            .flip(-1)
            .contiguous()
        )

        plus_intensity_target_gpu = (
            torch.log1p(plus_count)
            .to(device)
        )
        minus_intensity_target_reverse_gpu = (
            torch.log1p(minus_count_reverse)
            .to(device)
        )

        plus_intensity_weight_gpu = (
            2.0
            * batch.plus_loss_weight
            * (plus_count > 0).float()
        ).to(device)

        minus_intensity_weight_reverse_gpu = (
            2.0
            * batch.minus_loss_weight_reverse
            * (minus_count_reverse > 0).float()
        ).to(device)

        exp002_optimizer.zero_grad(
            set_to_none=True
        )

        (
            plus_presence,
            minus_presence_reverse,
            plus_intensity,
            minus_intensity_reverse,
        ) = forward_both_strands_multitask(
            exp002_model,
            x_gpu,
            batch_size=EXP006_BATCH_SIZE,
            context_flank=(
                profile_config.context_flank
            ),
            target_length=(
                profile_config.target_length
            ),
        )

        presence_loss = (
            dml_weighted_profile_bce_with_logits(
                plus_presence,
                minus_presence_reverse,
                plus_target_gpu,
                minus_target_reverse_gpu,
                plus_presence_weight_gpu,
                minus_presence_weight_reverse_gpu,
            )
        )

        intensity_loss = (
            dml_weighted_huber(
                plus_intensity,
                plus_intensity_target_gpu,
                plus_intensity_weight_gpu,
                delta=1.0,
            )
            + dml_weighted_huber(
                minus_intensity_reverse,
                minus_intensity_target_reverse_gpu,
                minus_intensity_weight_reverse_gpu,
                delta=1.0,
            )
        )

        total_loss = (
            presence_loss
            + INTENSITY_LOSS_WEIGHT
            * intensity_loss
        )

        presence_value = presence_loss.item()
        intensity_value = intensity_loss.item()
        total_value = total_loss.item()

        if not all(
            math.isfinite(value)
            for value in (
                presence_value,
                intensity_value,
                total_value,
            )
        ):
            raise FloatingPointError(
                f"Non-finite loss at step {step}"
            )

        total_loss.backward()
        exp002_optimizer.step()

        if ema_total is None:
            ema_presence = presence_value
            ema_intensity = intensity_value
            ema_total = total_value
        else:
            ema_presence = (
                0.99 * ema_presence
                + 0.01 * presence_value
            )
            ema_intensity = (
                0.99 * ema_intensity
                + 0.01 * intensity_value
            )
            ema_total = (
                0.99 * ema_total
                + 0.01 * total_value
            )

        seen_position_strand += int(
            batch.mask.sum()
        )

        if (
            step == 1
            or step % EXP006_LOG_EVERY == 0
        ):
            elapsed = (
                time.perf_counter()
                - started_at
            )

            record = {
                "step": step,
                "presence_loss": presence_value,
                "intensity_loss": intensity_value,
                "total_loss": total_value,
                "ema_presence": ema_presence,
                "ema_intensity": ema_intensity,
                "ema_total": ema_total,
                "elapsed_seconds": elapsed,
                "seen_position_strand": (
                    seen_position_strand
                ),
                "positions_per_second": (
                    seen_position_strand / elapsed
                ),
            }

            exp002_history.append(record)

            print(
                f"step={step:5d} "
                f"presence={presence_value:.4f} "
                f"intensity={intensity_value:.4f} "
                f"total={total_value:.4f} "
                f"ema=({ema_presence:.4f}, "
                f"{ema_intensity:.4f}, "
                f"{ema_total:.4f}) "
                f"positions/s="
                f"{seen_position_strand / elapsed:,.0f}"
            )

        if step in EXP006_CHECKPOINT_STEPS:
            checkpoint_path = (
                EXP006_DIR
                / f"model_step_{step:05d}.pt"
            )

            save_exp002_checkpoint(
                checkpoint_path,
                step=step,
                elapsed_seconds=(
                    time.perf_counter()
                    - started_at
                ),
                seen_position_strand=(
                    seen_position_strand
                ),
                history=exp002_history,
            )

fallback_warnings = [
    str(item.message)
    for item in caught
    if "fall back" in str(item.message).lower()
]

elapsed = time.perf_counter() - started_at

final_checkpoint_path = (
    EXP006_DIR / "exp002_final.pt"
)
history_path = (
    EXP006_DIR / "training_history.csv"
)

save_exp002_checkpoint(
    final_checkpoint_path,
    step=EXP006_STEPS,
    elapsed_seconds=elapsed,
    seen_position_strand=seen_position_strand,
    history=exp002_history,
)

pd.DataFrame(exp002_history).to_csv(
    history_path,
    index=False,
)

print()
print("Общее время:", round(elapsed, 2), "сек")
print(
    "Время шага:",
    round(elapsed / EXP006_STEPS, 4),
    "сек",
)
print(
    "Увидено position-strand:",
    seen_position_strand,
)
print(
    "Финальный EMA presence:",
    round(ema_presence, 6),
)
print(
    "Финальный EMA intensity:",
    round(ema_intensity, 6),
)
print(
    "Финальный EMA total:",
    round(ema_total, 6),
)
print("CPU fallback:", fallback_warnings)
print("Checkpoint:", final_checkpoint_path)
print("История:", history_path)

assert not fallback_warnings
assert final_checkpoint_path.is_file()
assert history_path.is_file()

In [ ]:
from ml_project.epic_cnn.evaluation import (
    evaluate_presence_intensity_model,
    load_cnn_presence_intensity_checkpoint,
)


EXP006_CHECKPOINT_PATH = (
    PROJECT_ROOT
    / "artifacts/experiments/CNN_EXP006"
    / "run_001"
    / "exp002_final.pt"
)

loaded_exp002 = (
    load_cnn_presence_intensity_checkpoint(
        EXP006_CHECKPOINT_PATH,
        device=device,
    )
)

validation_model = loaded_exp002.model
validation_config = loaded_exp002.profile_config
checkpoint = loaded_exp002.checkpoint

validation_generator = ProfileWindowGenerator(
    prepared_split=split,
    sequences=sequences,
    part="validation",
    config=validation_config,
    seed=42,
)

print("Checkpoint step:", checkpoint["step"])
print("Validation tiles:", len(validation_generator.tile_index))
print(
    "Validation position-strand:",
    int(
        validation_generator
        .tile_index["allowed_position_strand"]
        .sum()
    ),
)
print(
    "Validation positives:",
    validation_generator.total_positive,
)
print(
    "Device:",
    next(validation_model.parameters()).device,
)

In [ ]:
validation_result = (
    evaluate_presence_intensity_model(
        validation_model,
        validation_generator,
        device=device,
        batch_size=8,
        score_decimals=5,
        progress_every_batches=500,
    )
)

In [ ]:
VALIDATION_METRICS_PATH = (
    EXP006_CHECKPOINT_PATH.parent
    / "validation_metrics.csv"
)

validation_result.metrics.to_csv(
    VALIDATION_METRICS_PATH,
    index=False,
)

overall_metrics = (
    validation_result.metrics
    .loc[
        validation_result.metrics["segment"]
        == "overall"
    ]
    .reset_index(drop=True)
)

display(overall_metrics)

print(
    "Validation time:",
    round(validation_result.elapsed_seconds, 2),
    "sec",
)
print(
    "Position-strand/s:",
    f"{validation_result.positions_per_second:,.0f}",
)
print(
    "CPU fallback:",
    list(validation_result.fallback_warnings),
)
print("Metrics:", VALIDATION_METRICS_PATH)

In [ ]:
per_contig_metrics = (
    validation_result.metrics
    .loc[
        validation_result.metrics["segment"]
        != "overall"
    ]
    .sort_values(
        ["segment", "score"]
    )
    .reset_index(drop=True)
)

display(per_contig_metrics)

In [ ]:
# Запускай эту ячейку только после обучения и полной validation.
SAVE_EXPERIMENT = True
FINAL_DECISION = "adopt"  # review / adopt / reject / iterate
FINAL_NOTES = "RF++ охват очень сильно влияет нужно смотреть шире!"

if SAVE_EXPERIMENT:
    from ml_project.cnn_experiment_report import finalize_cnn_experiment

    finalization = finalize_cnn_experiment(
        project_root=PROJECT_ROOT,
        experiment_id='CNN-EXP-006',
        title='RF++',
        hypothesis='увеличиваю ещё сильнее RF до максимума',
        notebook_path='notebooks/experiments/CNN_experiments/CNN_EXP006_rf.ipynb',
        artifact_dir=PROJECT_ROOT / 'artifacts/experiments/CNN_EXP006/run_001',
        training_history=history_path,
        candidate_metrics=VALIDATION_METRICS_PATH,
        reference_experiment='CNN-EXP-003',
        reference_metrics='artifacts/experiments/CNN_EXP003/run_001/validation_metrics.csv',
        decision=FINAL_DECISION,
        notes=FINAL_NOTES,
    )
    print("Obsidian:", finalization.card_path)
    print("Summary:", finalization.summary_path)
    print("Plots:", finalization.plot_dir)
else:
    print("Сохранение выключено. Установи SAVE_EXPERIMENT = True после validation.")

In [ ]:
# Запускай эту ячейку только после обучения и полной validation.
SAVE_EXPERIMENT = True
FINAL_DECISION = "adopt"  # review / adopt / reject / iterate
FINAL_NOTES = "увеличенное число шагов сильно улучшает AP"

if SAVE_EXPERIMENT:
    from ml_project.cnn_experiment_report import finalize_cnn_experiment

    finalization = finalize_cnn_experiment(
        project_root=PROJECT_ROOT,
        experiment_id='CNN-EXP-006',
        title='STEP++ exp',
        hypothesis='увеличение шагов даст  явный прирост AP',
        notebook_path='notebooks/experiments/CNN_experiments/CNN_EXP006_step_exp.ipynb',
        artifact_dir=PROJECT_ROOT / 'artifacts/experiments/CNN_EXP006/run_001',
        training_history=history_path,
        candidate_metrics=VALIDATION_METRICS_PATH,
        reference_experiment='CNN-EXP-004',
        reference_metrics='artifacts/experiments/CNN_EXP004/run_001/validation_metrics.csv',
        decision=FINAL_DECISION,
        notes=FINAL_NOTES,
    )
    print("Obsidian:", finalization.card_path)
    print("Summary:", finalization.summary_path)
    print("Plots:", finalization.plot_dir)
else:
    print("Сохранение выключено. Установи SAVE_EXPERIMENT = True после validation.")

In [ ]:
# Запускай эту ячейку только после обучения и полной validation.
SAVE_EXPERIMENT = False
FINAL_DECISION = "review"  # review / adopt / reject / iterate
FINAL_NOTES = ""

if SAVE_EXPERIMENT:
    from ml_project.cnn_experiment_report import finalize_cnn_experiment

    finalization = finalize_cnn_experiment(
        project_root=PROJECT_ROOT,
        experiment_id='CNN-EXP-006',
        title='BPE test',
        hypothesis='Хочется посмотреть сколько даст сжатая последовательность',
        notebook_path='notebooks/experiments/CNN_experiments/CNN_EXP006_bpe_test.ipynb',
        artifact_dir=PROJECT_ROOT / 'artifacts/experiments/CNN_EXP006/run_001',
        training_history=history_path,
        candidate_metrics=VALIDATION_METRICS_PATH,
        reference_experiment='CNN-EXP-005',
        reference_metrics='artifacts/experiments/CNN_EXP005/run_001/validation_metrics.csv',
        decision=FINAL_DECISION,
        notes=FINAL_NOTES,
    )
    print("Obsidian:", finalization.card_path)
    print("Summary:", finalization.summary_path)
    print("Plots:", finalization.plot_dir)
else:
    print("Сохранение выключено. Установи SAVE_EXPERIMENT = True после validation.")